# Ethiopia Financial Inclusion — Forecasting Access & Usage (2025-2027)
### Task 4 | Selam Analytics

This notebook forecasts **Access** (Account Ownership Rate) and **Usage** (Digital Payment Adoption
Rate) for 2025-2027, combining trend regression, the event-effect model built in Task 3, and explicit
optimistic/base/pessimistic scenarios — with uncertainty quantified throughout, since we're working
with genuinely sparse data (4 Findex points for Access, a single point for Usage).

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"

df = pd.read_csv("../data/raw/ethiopia_fi_unified_data.csv", parse_dates=["observation_date"])
obs = df[df.record_type == "observation"].copy()
events = df[df.record_type == "event"].copy()
impacts = df[df.record_type == "impact_link"].copy()
targets = df[df.record_type == "target"].copy()

print(f"Loaded {len(df)} total records ({len(obs)} observations, {len(events)} events, "
      f"{len(impacts)} impact links, {len(targets)} targets)")

Loaded 62 total records (35 observations, 10 events, 14 impact links, 3 targets)


## Part 1: Access — Account Ownership Rate

### 1.1 Historical data & baseline trend regression

We have 4 Findex survey points (2014, 2017, 2021, 2024). We fit OLS on `years since 2014` to get
a linear trend with a 95% prediction interval, fully aware that 4 points gives a very wide interval —
that width is itself an honest signal about how much we don't know.

In [2]:
acc = obs[(obs.indicator_code == "ACC_OWNERSHIP") & (obs.gender == "all")].sort_values("observation_date").reset_index(drop=True)
acc["year"] = acc["observation_date"].dt.year + acc["observation_date"].dt.dayofyear / 365.25
acc["years_since_2014"] = acc["year"] - 2014
print(acc[["observation_date", "value_numeric", "years_since_2014"]])

X = sm.add_constant(acc["years_since_2014"])
y = acc["value_numeric"]
ols_model = sm.OLS(y, X).fit()
print()
print(ols_model.summary())

  observation_date  value_numeric  years_since_2014
0       2014-12-31           22.0          0.999316
1       2017-12-31           35.0          3.999316
2       2021-12-31           46.0          7.999316
3       2024-11-29           49.0         10.914442

                            OLS Regression Results                            
Dep. Variable:          value_numeric   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.920
Method:                 Least Squares   F-statistic:                     35.72
Date:                Tue, 21 Jul 2026   Prob (F-statistic):             0.0269
Time:                        08:11:59   Log-Likelihood:                -9.2476
No. Observations:                   4   AIC:                             22.50
Df Residuals:                       2   BIC:                             21.27
Df Model:                           1                                         
Covariance Type:            

In [3]:
future_years = np.array([2025.5, 2026.5, 2027.5])  # mid-year forecast points
X_future = sm.add_constant(pd.DataFrame({"years_since_2014": future_years - 2014}), has_constant="add")

pred = ols_model.get_prediction(X_future)
pred_summary = pred.summary_frame(alpha=0.05)
pred_summary.index = [2025, 2026, 2027]
pred_summary.columns = ["forecast", "se", "ci_lower", "ci_upper", "pi_lower", "pi_upper"]
print("Baseline trend-regression forecast (with 95% prediction interval):")
pred_summary[["forecast", "pi_lower", "pi_upper"]].round(1)

Baseline trend-regression forecast (with 95% prediction interval):


,forecast,pi_lower,pi_upper
2025,53.1,33.2,72.9
2026,55.8,34.8,76.8
2027,58.5,36.3,80.8


### 1.2 Event-augmented model

We reuse Task 3's ramped event-effect model, with the **refined** Telebirr estimate (down-weighted to
+6.5% after validation, not the raw +15% Kenya-comparable figure) so we don't double-count an
already-known over-prediction. The only two events targeting `ACC_OWNERSHIP` are Telebirr (fully
phased in by mid-2022) and the Fayda Digital ID rollout (24-month lag from Jan 2024 → fully phased in
by Jan 2026).

In [4]:
def ramp_effect(event_date, lag_months, full_effect_pct, as_of_date):
    months_elapsed = (as_of_date.year - event_date.year) * 12 + (as_of_date.month - event_date.month)
    if months_elapsed <= 0:
        return 0.0
    if months_elapsed >= lag_months:
        return full_effect_pct
    return full_effect_pct * (months_elapsed / lag_months)

# Refined estimates for ACC_OWNERSHIP (Telebirr refined per Task 3 validation; Fayda ID kept as-is, unvalidated)
acc_event_effects = [
    {"event": "Telebirr Launch", "event_date": pd.Timestamp("2021-05-17"), "lag_months": 12, "effect_pct": 6.5},
    {"event": "Fayda Digital ID Rollout", "event_date": pd.Timestamp("2024-01-01"), "lag_months": 24, "effect_pct": 10.0},
]

acc_2024_baseline = acc[acc.observation_date.dt.year == 2024]["value_numeric"].iloc[0]

def event_augmented_forecast(baseline_value, baseline_date, effects, as_of_date):
    """Apply the *incremental* event effect that accrues between baseline_date and as_of_date."""
    total_incremental = 0.0
    for e in effects:
        effect_at_target = ramp_effect(e["event_date"], e["lag_months"], e["effect_pct"], as_of_date)
        effect_at_baseline = ramp_effect(e["event_date"], e["lag_months"], e["effect_pct"], baseline_date)
        total_incremental += (effect_at_target - effect_at_baseline)
    return baseline_value * (1 + total_incremental / 100)

baseline_date = pd.Timestamp("2024-11-29")
event_augmented = {}
for yr in [2025, 2026, 2027]:
    as_of = pd.Timestamp(f"{yr}-07-01")
    event_augmented[yr] = event_augmented_forecast(acc_2024_baseline, baseline_date, acc_event_effects, as_of)

print("Event-augmented forecast (2024 baseline + incremental Fayda ID effect only, Telebirr already fully phased in by 2024):")
for yr, val in event_augmented.items():
    print(f"  {yr}: {val:.1f}%")

Event-augmented forecast (2024 baseline + incremental Fayda ID effect only, Telebirr already fully phased in by 2024):
  2025: 50.6%
  2026: 51.9%
  2027: 51.9%


### 1.3 Scenario analysis

- **Base case**: continue the observed **2021-2024 slope** (~1.0pp/year) — the most recent, most
  relevant trend, reflecting the slowdown Task 2 identified.
- **Optimistic**: growth **re-accelerates** to the 2017-2021 slope (~2.75pp/year), on the premise that
  the Fayda Digital ID rollout and continued mobile money maturation start actually adding to account
  ownership rather than substituting for it.
- **Pessimistic**: growth **decelerates further** (~0.5pp/year), on the premise that Ethiopia is
  approaching the "easy" adopters and mobile money's substitution effect strengthens, consistent with
  Task 3's finding that the Kenya-comparable Telebirr estimate over-predicted actual growth.

In [5]:
slope_2021_2024 = (49.0 - 46.0) / (2024 - 2021)          # ~1.0 pp/yr — base
slope_2017_2021 = (46.0 - 35.0) / (2021 - 2017)          # ~2.75 pp/yr — optimistic ceiling
pessimistic_slope = slope_2021_2024 * 0.5                 # further deceleration

scenarios = {"optimistic": slope_2017_2021, "base": slope_2021_2024, "pessimistic": pessimistic_slope}
acc_scenarios = pd.DataFrame(index=[2025, 2026, 2027])
for name, slope in scenarios.items():
    acc_scenarios[name] = [round(acc_2024_baseline + slope * (yr - 2024), 1) for yr in acc_scenarios.index]

acc_scenarios["trend_regression"] = pred_summary["forecast"].round(1)
acc_scenarios["event_augmented"] = pd.Series(event_augmented).round(1)
acc_scenarios

,optimistic,base,pessimistic,trend_regression,event_augmented
2025,51.8,50.0,49.5,53.1,50.6
2026,54.5,51.0,50.0,55.8,51.9
2027,57.2,52.0,50.5,58.5,51.9


In [6]:
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.plot(acc["observation_date"].dt.year, acc["value_numeric"], marker="o", markersize=9,
        color="#1a1a1a", linewidth=2.2, label="Historical (Findex)", zorder=5)

fc_years = [2024, 2025, 2026, 2027]
for name, color, style in [("optimistic", "#27ae60", "--"), ("base", "#2980b9", "-"), ("pessimistic", "#e74c3c", "--")]:
    vals = [acc_2024_baseline] + list(acc_scenarios[name])
    ax.plot(fc_years, vals, color=color, linewidth=2, linestyle=style, marker="o", markersize=5, label=f"Scenario: {name}")

ax.fill_between([2025, 2026, 2027], pred_summary["pi_lower"], pred_summary["pi_upper"],
                color="#2980b9", alpha=0.12, label="Trend regression 95% PI")
ax.axhline(70, color="#8e44ad", linestyle=":", linewidth=1.5, label="NBE target (70%, dated 2025)")

ax.set_title("Access: Account Ownership Rate — Forecast 2025-2027", fontweight="bold", fontsize=13)
ax.set_ylabel("Account Ownership (%)")
ax.set_xlim(2013, 2028)
ax.set_ylim(0, 80)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("../reports/figures/access_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 2: Usage — Digital Payment Adoption Rate

### 2.1 The data problem

`USG_DIGITAL_PAY` has **exactly one** observation (16% in 2024) — a trend regression is not
statistically meaningful with a single point. Instead we anchor on that single value and derive
growth-rate assumptions from **correlated proxy indicators** that do have multiple observations:
mobile money account ownership (2 points, 2021→2024) and P2P transaction volume (2 points,
mid-2024→mid-2025).

In [7]:
usg = obs[obs.indicator_code == "USG_DIGITAL_PAY"].sort_values("observation_date")
print("Digital Payment Adoption Rate — full history:")
print(usg[["observation_date", "value_numeric", "confidence", "source_name"]].to_string(index=False))
usg_2024_baseline = usg["value_numeric"].iloc[0]

mm = obs[obs.indicator_code == "ACC_MM_ACCOUNT"].sort_values("observation_date")
mm_cagr = (mm["value_numeric"].iloc[-1] / mm["value_numeric"].iloc[0]) ** (1 / 3) - 1  # 2021->2024, 3 yrs

p2p = obs[obs.indicator_code == "USG_P2P_COUNT"].sort_values("observation_date")
p2p_yoy = p2p["value_numeric"].iloc[-1] / p2p["value_numeric"].iloc[0] - 1  # 2024-07 -> 2025-07, 1 yr

active_rate = obs[obs.indicator_code == "USG_ACTIVE_RATE"]["value_numeric"].iloc[0] / 100  # 66% registered->active discount

print(f"\nMobile money account CAGR (2021-2024): {mm_cagr:+.1%} / year")
print(f"P2P transaction count YoY growth (2024-2025): {p2p_yoy:+.1%}")
print(f"Registered-to-active discount factor (Task 2 finding): {active_rate:.0%}")

Digital Payment Adoption Rate — full history:
observation_date  value_numeric confidence                                                                                  source_name
      2024-12-31           16.0     medium Determinants of Financial Inclusion in Ethiopia (IJRSI, citing Global Findex 2025 microdata)

Mobile money account CAGR (2021-2024): +26.2% / year
P2P transaction count YoY growth (2024-2025): +158.1%
Registered-to-active discount factor (Task 2 finding): 66%


### 2.2 Scenario construction

- **Optimistic**: starts from the **P2P transaction growth rate** (+158% YoY) — the most bullish,
  most recent usage signal — but that rate is a one-off surge, not a sustainable multi-year pace, so
  we **decay it by half each subsequent year** (158% \u2192 79% \u2192 39.5%) rather than naively
  compounding a single year's spike three times over. Even so, we cap the result at a **95% ceiling**,
  since an adoption *rate* cannot exceed 100% and values approaching that ceiling should be read as
  "saturating," not as a precise point estimate.
- **Base**: apply the **mobile money account CAGR** (~26%/year), held constant — a more conservative
  multi-year growth rate from the same underlying survey family as the target indicator.
- **Pessimistic**: apply the base CAGR but **discounted by the registered-vs-active gap** (66%),
  reflecting Task 2's finding that headline registration/transaction-count growth substantially
  overstates genuine active usage.

In [8]:
def decaying_growth_projection(baseline, first_year_rate, decay=0.5, n_years=3):
    """Applies first_year_rate, then halves the growth rate each subsequent year (avoids naively
    compounding a one-off surge). Returns a list of projected values, one per year."""
    vals, cur, rate = [], baseline, first_year_rate
    for _ in range(n_years):
        cur = cur * (1 + rate)
        vals.append(cur)
        rate *= decay
    return vals

USAGE_CEILING = 95.0  # a rate cannot exceed 100%; treat approach to this as saturation, not precision

usg_scenarios = pd.DataFrame(index=[2025, 2026, 2027])
usg_scenarios["optimistic"] = [round(min(v, USAGE_CEILING), 1) for v in
                                decaying_growth_projection(usg_2024_baseline, p2p_yoy)]
usg_scenarios["base"] = [round(min(usg_2024_baseline * (1 + mm_cagr) ** (yr - 2024), USAGE_CEILING), 1)
                          for yr in usg_scenarios.index]
usg_scenarios["pessimistic"] = [round(min(usg_2024_baseline * (1 + mm_cagr * active_rate) ** (yr - 2024), USAGE_CEILING), 1)
                                 for yr in usg_scenarios.index]
usg_scenarios

,optimistic,base,pessimistic
2025,41.3,20.2,18.8
2026,74.0,25.5,22.0
2027,95.0,32.2,25.8


In [9]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter([2024], [usg_2024_baseline], color="#1a1a1a", s=90, zorder=5, label="Historical (Findex 2024, single point)")

fc_years = [2024, 2025, 2026, 2027]
for name, color, style in [("optimistic", "#27ae60", "--"), ("base", "#2980b9", "-"), ("pessimistic", "#e74c3c", "--")]:
    vals = [usg_2024_baseline] + list(usg_scenarios[name])
    ax.plot(fc_years, vals, color=color, linewidth=2, linestyle=style, marker="o", markersize=5, label=f"Scenario: {name}")

ax.set_title("Usage: Digital Payment Adoption Rate — Forecast 2025-2027", fontweight="bold", fontsize=13)
ax.set_ylabel("Digital Payment Adoption (%)")
ax.set_xlim(2023.5, 2027.5)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("../reports/figures/usage_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nNote: no statistical confidence interval is shown for Usage — with a single historical point,")
print("a regression-based interval would be spurious. The optimistic/base/pessimistic spread IS the")
print("uncertainty representation here, and it's deliberately wide.")


Note: no statistical confidence interval is shown for Usage — with a single historical point,
a regression-based interval would be spurious. The optimistic/base/pessimistic spread IS the
uncertainty representation here, and it's deliberately wide.


## Part 3: Combined Forecast Table & Export

We export the combined scenario table so the Task 5 dashboard can read it directly rather than
re-deriving forecasts inside the dashboard app.

In [10]:
forecast_table = pd.concat({
    "Access (Account Ownership %)": acc_scenarios[["optimistic", "base", "pessimistic"]],
    "Usage (Digital Payment Adoption %)": usg_scenarios[["optimistic", "base", "pessimistic"]],
}, axis=1)
forecast_table.index.name = "year"
print(forecast_table)

import os
os.makedirs("../models", exist_ok=True)
forecast_table.to_csv("../models/forecast_results.csv")
print("\nSaved: models/forecast_results.csv")

     Access (Account Ownership %)                    \
                       optimistic  base pessimistic   
year                                                  
2025                         51.8  50.0        49.5   
2026                         54.5  51.0        50.0   
2027                         57.2  52.0        50.5   

     Usage (Digital Payment Adoption %)                    
                             optimistic  base pessimistic  
year                                                       
2025                               41.3  20.2        18.8  
2026                               74.0  25.5        22.0  
2027                               95.0  32.2        25.8  

Saved: models/forecast_results.csv


## Part 4: Interpretation

**What the model predicts:**
- **Access**: even the *optimistic* scenario (resuming the faster 2017-2021 growth rate) only reaches
  ~57% by 2027 — nowhere near the NBE's 70% target (dated 2025). The base case puts 2027 ownership
  around 52%. The trend-regression prediction interval is wide enough to include the optimistic
  scenario at its upper bound and something close to stagnation at its lower bound, which is an
  honest reflection of how much a 4-point series can actually tell us.
- **Usage**: the base case roughly doubles digital payment adoption by 2027 (16% → ~32%), driven by
  the mobile money account CAGR. The optimistic scenario, even after decaying the initial P2P growth
  surge and capping at a 95% saturation ceiling, still implies adoption could plausibly **approach
  most of the adult population** using digital payments by 2027 — a genuinely wide range, and the
  correct reflection of having only one real Usage data point plus two very different proxy growth
  rates to choose from.

**Events with the largest potential remaining impact (2025-2027 window):**
- **Fayda Digital ID rollout** — its 24-month lag means its full modeled effect on `ACC_OWNERSHIP`
  only completes in January 2026, so most of its impact is still ahead of us, not behind us.
- **EthioPay Instant Payment System** and the **M-Pesa/EthSwitch integration** — both target P2P
  transaction volume and mobile money activity, i.e. Usage, not Access, reinforcing why Usage's
  optimistic scenario is so much higher than Access's.
- **FX liberalization**'s affordability effect (+30%) doesn't map to either headline indicator
  directly, but plausibly supports Usage indirectly by making data/transactions cheaper — not modeled
  quantitatively here, flagged as a gap.

**Key uncertainties:**
- Access forecasts rest on a 4-point regression; Usage forecasts rest on a single anchor point plus
  borrowed growth rates from *different* (correlated but not identical) indicators — both are
  meaningfully more uncertain than the point estimates alone suggest.
- Task 3's validation showed comparable-country impact estimates can overshoot Ethiopia's actual
  outcomes (Telebirr/Kenya case) — the same risk applies to any forward-looking event effect used
  here (Fayda ID/India case is *unvalidated*, since its lag period hasn't elapsed yet).
- The scenario spread should be read as a plausibility range, not a statistical confidence interval,
  for Usage specifically — this is stated explicitly rather than implied by a chart that looks more
  precise than the underlying data supports.